# 10 — Statistical Power Simulation (n=207)

Continuation of notebooks 01–06. Simulates METR-LA-scale data (n=207 sensors) repeatedly through the real `faircause` estimator to check whether this sample size gives stable, well-calibrated confidence intervals for the Ctf-IE effects.


In [ ]:
# --- R environment setup for this notebook (safe to re-run; skips if already installed) ---
import subprocess
import sys
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base-core"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
get_ipython().run_line_magic("load_ext", "rpy2.ipython")
print("R + rpy2 bridge ready.")


In [ ]:
%%R
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA")
library(faircause)
cat("faircause loaded, version:", as.character(packageVersion("faircause")), "\n")


In [ ]:
%%R
#!/usr/bin/env Rscript
# ============================================================================
# 03_power_simulation.R
#
# Run in Colab via %%R after notebooks 00 and 02 have confirmed faircause
# works. NOT executed/tested locally. This simulates n=207 data repeatedly
# through the ACTUAL faircause estimator (not a generic power formula) to
# see how often it recovers a known true effect and how wide its confidence
# intervals typically are at this sample size.
# ============================================================================

library(faircause)

1. Data-generating function matching the project DAG, with a KNOWN true
   Ctf-IE for reliability (0.5) that we'll check the estimator recovers

In [ ]:
%%R
real_data <- read.csv("metr_la_metrics.csv")
real_data$reliability <- pmax(0, 1.0 - (0.6 * real_data$zero_rate + 0.2 * real_data$cusum_flag_rate + 0.2 * real_data$ewma_flag_rate))
real_data$disparity <- real_data$persistence_error

simulate_once <- function(n = 207, true_reliability_effect = 0.5) {
  idx <- sample(1:nrow(real_data), n, replace = TRUE)
  df <- real_data[idx, ]
  
  df$disparity <- true_reliability_effect * (1 - df$reliability) + 
                  0.01 * df$topology + 
                  0.01 * df$density + 
                  rnorm(n, sd = sd(real_data$disparity)*0.1)

  df$density_bin <- ifelse(df$density > median(df$density), "high_density", "low_density")
  return(df)
}


2. Run N_SIM replicates, record whether the CI covers the true effect and
   how wide the CI is

In [ ]:
%%R
N_SIM <- 500  # reduce to ~50 for a quick first test before committing to 500
TRUE_EFFECT <- 0.5

coverage <- logical(N_SIM)
ci_widths <- numeric(N_SIM)
point_estimates <- numeric(N_SIM)

for (i in seq_len(N_SIM)) {
  sim_data <- simulate_once(n = 207, true_reliability_effect = TRUE_EFFECT)

  result <- tryCatch({
    fairness_cookbook(
      data = sim_data,
      X = "density_bin",
      Z = c("traffic_regime", "road_type"),
      W = c("reliability", "topology"),
      Y = "disparity",
      x0 = "low_density", x1 = "high_density"
    )
  }, error = function(e) NULL)

  if (is.null(result)) {
    coverage[i] <- NA
    ci_widths[i] <- NA
    point_estimates[i] <- NA
    next
  }

  # Extract the indirect effect estimate for reliability and its CI.
  # ADJUST field names below once you confirm the actual structure of
  # `result` / `summary(result)` from running 02_ctf_estimation_faircause.R —
  # this is a best-guess based on typical faircause output structure and
  # needs verification against the real object.
  s <- summary(result)
  # placeholder extraction logic:
  ie_row <- s[s$measure == "Ctf-IE" & grepl("reliability", s$mediator), ]
  if (nrow(ie_row) == 0) {
    coverage[i] <- NA; ci_widths[i] <- NA; point_estimates[i] <- NA
    next
  }
  point_estimates[i] <- ie_row$value[1]
  ci_lo <- ie_row$ci_lower[1]
  ci_hi <- ie_row$ci_upper[1]
  ci_widths[i] <- ci_hi - ci_lo
  coverage[i] <- (TRUE_EFFECT >= ci_lo) && (TRUE_EFFECT <= ci_hi)
}

3. Summarize

In [ ]:
%%R
n_valid <- sum(!is.na(coverage))
cat("=== Power simulation results (n=207, N_SIM =", N_SIM, ") ===\n")
cat("Successful runs:", n_valid, "/", N_SIM, "\n")
if (n_valid > 0) {
  cat("Empirical CI coverage of true effect (target ~95%):",
      round(100 * mean(coverage, na.rm = TRUE), 1), "%\n")
  cat("Mean CI width:", round(mean(ci_widths, na.rm = TRUE), 3), "\n")
  cat("Mean point estimate (true value =", TRUE_EFFECT, "):",
      round(mean(point_estimates, na.rm = TRUE), 3), "\n")
  cat("SD of point estimates (estimator variability at n=207):",
      round(sd(point_estimates, na.rm = TRUE), 3), "\n")
}

cat("\n=== Interpretation ===\n")
cat("If coverage is well below 95%, the estimator's CIs are too narrow for\n")
cat("this sample size (overconfident). If mean CI width is large relative to\n")
cat("the effect size itself, n=207 may be too small for a precise estimate\n")
cat("even if coverage is technically correct. Either signals a need for a\n")
cat("simpler, more stable estimator (e.g. parametric mediation with linear/\n")
cat("logistic nuisance models) as a fallback for the real analysis.\n")

cat("\n=== NOTE ===\n")
cat("The field-name extraction logic in step 2 above (`ie_row <- s[...]`) is\n")
cat("a best guess and WILL likely need correcting once you see faircause's\n")
cat("actual summary() output structure from 02_ctf_estimation_faircause.R.\n")
cat("Run that script first, inspect `str(summary(result))`, then fix the\n")
cat("extraction logic here before trusting this simulation's numbers.\n")